<img src="../../../docs/assets/0.-BC-dev-hub-LOGO-flicker.svg" alt="BrainChip Dev Hub" width="200"/>

# Object Detection (YOLOv2 / PASCAL VOC) on Akida 1

<p align="right">
Run Time: several hours with training included / ~5 minutes with training skipped
</p>

This notebook steps through the full pipeline for a **YOLOv2 object detector**
(restricted to the 'car' and 'person' classes of **PASCAL VOC**) on **Akida 1
(AKD1500)**: training a tf_keras model, quantization and conversion to Akida
format, and evaluation of the resulting Akida model. For background on the
task, dataset, and model performance benchmarks, see the [README](README.md).

The focus is on the **Akida-specific** aspects of the pipeline. The full data
preprocessing and training code is available in the accompanying Python files
— most of it is reused directly from `akida_models.detection` and is not
described further in this notebook.

By default, model training is run to ensure reproducibility. However, you can
cut the running time of the notebook down if desired by skipping the training
runs and loading pretrained float and quantized models instead: simply set
the relevant `RUN_FLOAT_TRAINING` and `RUN_QAT_TRAINING` variables in the
first code cell to `False` (this requires the `pretrained_models/` folder
to be populated - see the [README](README.md)).

## Setup

The default dataset path is `./data/voc`. See the [README](README.md)
for download instructions. Update `DATA_PATH` below if needed.

In [ ]:
import os
import numpy as np
import tensorflow as tf

DATA_PATH = './data/voc'
MODELS_DIR = './models'
os.makedirs(MODELS_DIR, exist_ok=True)

RUN_FLOAT_TRAINING = True
RUN_QAT_TRAINING = True

SEED = 42

# Must be called before any TF ops to make GPU ops (conv backward passes,
# bilinear resize, etc.) deterministic. Has a small throughput cost.
tf.config.experimental.enable_op_determinism()

## Dataset

`get_data` returns a training and a validation `tf.data.Dataset`, each
yielding `(image, targets)` batches already encoded in the grid/anchor
format the YOLOv2 loss expects. `get_anchors` fetches the fixed set of
anchor boxes used to build these targets (and to decode predictions later) -
these must stay fixed for a given model, not be regenerated. We also load
the *raw* (unprocessed) validation data separately, which the mAP evaluator
needs directly. The full preprocessing/augmentation code is in
[detection_data.py](detection_data.py) and `akida_models.detection`.

In [ ]:
from detection_data import get_data, get_anchors, LABELS
from akida_models.detection.voc.data import get_voc_dataset

BATCH_SIZE = 32
INPUT_SHAPE = (224, 224, 3)

train_ds, val_ds, num_train = get_data(DATA_PATH, INPUT_SHAPE, BATCH_SIZE, seed=SEED)
anchors = get_anchors()

# Raw (unprocessed) validation data, used directly by the mAP evaluator below
val_data_raw, labels, num_valid = get_voc_dataset(DATA_PATH, labels=LABELS, training=False)

## Model

The backbone is **AkidaNet** — a MobileNet V1 variant whose layer structure
maps efficiently onto Akida hardware — with width multiplier `alpha=0.5`,
topped with a **YOLOv2** detection head.

Key design choices:

- **Input resolution 224x224**, matching the ImageNet-pretrained backbone
  used to initialize the model (only the backbone layer names match the
  pretrained weights file; the head is freshly initialized).
- **5 anchor boxes x (4 box coords + 1 objectness + 2 classes)** raw output
  per grid cell, over a 7x7 grid — the standard YOLOv2 output layout.
- The model embeds input scaling as its first layer, so inputs must be
  delivered as uint8 pixel values, not pre-normalized.
- **`AkidaVersion.v1` context** constrains the layer configuration to what
  the AKD1500 hardware supports.

In [ ]:
from detection_model import build_detection_model
model = build_detection_model(seed=SEED)
model.summary()

## Float Training

The YOLOv2 head is trained for 70 epochs using Adam at a fixed learning
rate, directly on the VOC ('car', 'person') subset. The full training code
is in [detection_train.py](detection_train.py) — standard tf_keras code
wrapping the `YoloLoss` from `akida_models.detection`, not described
further here.

In [ ]:
from detection_train import train_detection


if RUN_FLOAT_TRAINING:
    LEARNING_RATE = 5e-4
    EPOCHS = 70
    # Make sure the dataset seed is freshly set for reproducibility
    train_ds, val_ds, num_train = get_data(DATA_PATH, INPUT_SHAPE, BATCH_SIZE, seed=SEED)

    train_detection(model, train_ds, val_ds, num_train,
            EPOCHS,
            LEARNING_RATE,
            anchors,
            BATCH_SIZE,
            seed=SEED
            )

    float_model_path = os.path.join(MODELS_DIR, 'yolo_akidanet_detection.h5')
    model.save(float_model_path, include_optimizer=False)
    print(f'Float model saved to {float_model_path}')
else:
    from tf_keras.models import load_model
    print('Training skipped. Using pretrained model...')
    model_path = 'pretrained_models/yolo_akidanet_detection.h5'
    model = load_model(model_path)

### Evaluate float model

`MapEvaluation` expects a Keras model whose output is already reshaped to
`(grid_h, grid_w, num_anchors, 5+classes)` — the flat conv output needs
wrapping with the same `Reshape` used internally during training.

In [ ]:
from tf_keras import Model
from tf_keras.layers import Reshape
from akida_models.detection.map_evaluation import MapEvaluation

def wrap_for_eval(keras_model):
    grid_size = keras_model.output_shape[1:3]
    num_classes = keras_model.output_shape[-1] // len(anchors) - 5
    out = Reshape((grid_size[0], grid_size[1], len(anchors), 5 + num_classes),
                 name='YOLO_output')(keras_model.output)
    return Model(keras_model.input, out)

map_evaluator = MapEvaluation(wrap_for_eval(model), val_data_raw, num_valid, labels, anchors)
map_dict, _ = map_evaluator.evaluate_map()
float_map = sum(map_dict.values()) / len(map_dict)
print(f'Float mAP: {float_map:.4f}')

## Quantization

Akida 1 operates with integer weights and activations. We use `cnn2snn` to
quantize the float model to 4 bits for both weights and activations (8-bit
weights are enabled for the first layer only, which is also unusual in
receiving uint8 inputs):

Post-training quantization maps the float parameters to their nearest
representable integer values. Some mAP is typically lost in this step,
which the subsequent QAT pass recovers.

Note: the quantized model can be saved using the standard method. However,
for later reloading, because of the custom quantized layers in the model
we have to use the `load_quantized_model` function from cnn2snn (a wrapper
around the standard tf_keras loading function)

In [ ]:
from cnn2snn import quantize, load_quantized_model

quantized_model = quantize(
    model,
    input_weight_quantization=8,
    weight_quantization=4,
    activ_quantization=4,
)
ptq_model_path = os.path.join(MODELS_DIR, 'yolo_akidanet_detection_ptq.h5')
quantized_model.save(ptq_model_path, include_optimizer=False)

del quantized_model

quantized_model = load_quantized_model(ptq_model_path)

### Quantization-Aware Training (QAT)

A few epochs of fine-tuning sufficient to recover most of the mAP lost
during quantization, at a lower learning rate than the initial training.

Note that, although Quantization Aware Training can sound intimidating,
the model quantized via `cnn2snn` can simply be reinserted into the
same training function that was used for the initial float training.

In [ ]:

if RUN_QAT_TRAINING:
    QAT_LEARNING_RATE = 5e-5
    QAT_EPOCHS = 5

    # We re-fetch the dataset just for reproducibility vs the script version of the code
    train_ds, val_ds, num_train = get_data(DATA_PATH, INPUT_SHAPE, BATCH_SIZE, seed=SEED)

    train_detection(quantized_model, train_ds, val_ds, num_train,
            QAT_EPOCHS,
            QAT_LEARNING_RATE,
            anchors,
            BATCH_SIZE,
            )

    qat_model_path = os.path.join(MODELS_DIR, 'yolo_akidanet_detection_qat.h5')
    quantized_model.save(qat_model_path, include_optimizer=False)
    print(f'QAT model saved to {qat_model_path}')
else:
    print('QAT skipped. Loading pretrained model...')
    q_model_path = 'pretrained_models/yolo_akidanet_detection_qat.h5'
    quantized_model = load_quantized_model(q_model_path)

### Evaluate quantized model

In [ ]:
map_evaluator = MapEvaluation(wrap_for_eval(quantized_model), val_data_raw, num_valid, labels, anchors)
map_dict, _ = map_evaluator.evaluate_map()
qat_map = sum(map_dict.values()) / len(map_dict)
print(f'QAT mAP: {qat_map:.4f}')

## Conversion to Akida Format

`cnn2snn.convert` compiles the quantized Keras model into an Akida `.fbz`
model that can be loaded and executed directly on AKD1500 hardware.
The converter verifies hardware compatibility and maps each layer to its
corresponding Akida primitive.

In [ ]:
from cnn2snn import convert

akida_model = convert(quantized_model)

akida_model_path = os.path.join(MODELS_DIR, 'yolo_akidanet_detection_qat.fbz')
akida_model.save(akida_model_path)
print(f'Akida model saved to {akida_model_path}')
akida_model.summary()

## Evaluation of Akida Model

We now run mAP evaluation through the Akida model, to check that it is
comparable to that obtained from the quantized tf_keras model. Here, we
deliberately use the software backend (the default, since we do not check
for and map to a connected hardware device): this delivers a bit-accurate
simulation of the results that will be obtained when running the model on
hardware.

In the accompanying [detection_notebook_benchmark.ipynb](detection_notebook_benchmark.ipynb)
the same evaluation is run using the hardware backend (if, of course, a hardware Akida
device is connected), allowing you to confirm that the results are identical.

`MapEvaluation` reshapes the flat Akida output internally, so no manual
reshape/wrap is needed here (unlike the Keras evaluation cells above).

In [ ]:
map_evaluator = MapEvaluation(akida_model, val_data_raw, num_valid, labels, anchors,
                              is_keras_model=False)
map_dict, _ = map_evaluator.evaluate_map()
akida_map = sum(map_dict.values()) / len(map_dict)
print(f'Akida mAP: {akida_map:.4f}')

### Activation Sparsity

Akida hardware skips computation for zero-valued activations, so activation
sparsity directly reduces both energy consumption and inference latency.
Below we measure per-layer sparsity on a calibration batch drawn from the
validation set.

In [ ]:
from akida_models.sparsity import compute_sparsity
from brainchip_utils.plot_utils import pretty_print_sparsity
from detection_data import get_samples

NUM_SAMPLES = 1000
samples = get_samples(DATA_PATH, INPUT_SHAPE, num_samples=NUM_SAMPLES)
sparsity_dict = compute_sparsity(akida_model, samples=samples)
pretty_print_sparsity(sparsity_dict)

## Summary


In [ ]:
print(f"Float mAP: {float_map:.4f}")
print(f"QAT mAP:   {qat_map:.4f}")
print(f"Akida mAP: {akida_map:.4f}")